# Ataques sobre publicaciones existentes
Ejecuta sin repetir k-scale. Consulta README_ATTACKS.md para los supuestos y límites.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from kscale import simulate_attacks, plot_attack_heatmap, single_point_compatible_values

output_path = Path("run_metrics_01")
Sigma = output_path / "Sigma"
k_values = [2, 5, 10]
generalized_paths = {k: output_path / f"k_{k}" / "Sigma_tilde" for k in k_values}
c_values = [1, 2, 3, 5]
beta_values = [1, 2, "all"]  # Use [1,2,4,8,"all"] for longer trajectories.
attack_trials = 100
attack_seed = 42
attack_name = "attacks_01"  # New directory for each new execution.


In [ ]:
manifest = pd.read_parquet(Sigma / "manifest.parquet")
min_points = max(b for b in beta_values if isinstance(b, int))
print("Eligible users:", int((manifest.N_i >= min_points).sum()))
print(manifest.N_i.describe())


In [ ]:
attack_paths = {}
for k, release in generalized_paths.items():
    attack_paths[k] = simulate_attacks(
        Sigma, release, output_path / f"k_{k}" / attack_name,
        c_values=c_values, beta_values=beta_values,
        trials=attack_trials, seed=attack_seed,
        victim_modes=("random", "nearby"),
        point_modes=("random", "consecutive"),
        spatial_scale_m=1000, time_scale_s=3600,
    )
comparison = pd.concat([
    pd.read_csv(path / "attack_summary.csv") for path in attack_paths.values()
], ignore_index=True)
comparison


In [ ]:
for k, path in attack_paths.items():
    ax = plot_attack_heatmap(path / "attack_summary.csv", metric="p_collapse")
    ax.set_title(f"k={k}: P(0 < J < k), random victims and points")
    ax.figure.savefig(path / "collapse_heatmap.png", dpi=180, bbox_inches="tight")
    plt.show()


## Un punto por usuario: conocimiento complementario
Este ejemplo tiene una rejilla discreta y solo publica un intervalo. No supone reconstrucción de trayectorias completas.

In [ ]:
import numpy as np
grid = np.arange(11)
for known, unknown_count in [([0,10],1), ([0,4],1), ([0],2), ([5],2)]:
    possible = single_point_compatible_values(known, unknown_count, [[0],[10]], grid)
    print("Known:", known, "Unknown users:", unknown_count, "Possible C:", possible.ravel())
